In [0]:
import json
import os
from urllib.parse import urljoin

import pandas as pd
import requests

BASE_URL = 'https://cvijan.github.io/nss-api/v2/'
FILES_URL = 'https://cvijan.github.io/nss-api/files/'
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

In [0]:
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
from pprint import pprint

In [0]:
# 1/18

def get_json(url: str) -> dict:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return response.json()

In [0]:
# 2/18

def fetch_all(first_url: str) -> list:
    orders = [] 
    url = first_url
    while url:
        page = get_json(url)
        orders.extend(page['data'])
        url = urljoin(url, page['next']) if page['next'] else None
    return orders

In [0]:
# 3/18

def save_json(pages: list, path: str) -> None:
    with open(path, 'w') as f:
        json.dump(pages, f, ensure_ascii=False, indent=2)

In [0]:
# 4/18

def download_file(url: str, path: str) -> int:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(path, 'wb') as f:
        f.write(response.content)
    return len(response.content)

# download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR, 'teams.csv'))
# download_file((FILES_URL + 'products.csv'), os.path.join(RAW_DIR, 'products.csv'))
# download_file((FILES_URL + 'matches.parquet'), os.path.join(RAW_DIR, 'matches.parquet'))

In [0]:
# 5/18

def extract(raw_dir: str) -> dict:
    orders = fetch_all(BASE_URL + 'orders/page-1.json')

    save_json(orders, os.path.join(raw_dir, 'orders.json'))    
    download_file(FILES_URL + 'teams.csv', os.path.join(raw_dir, 'teams.csv'))
    download_file(FILES_URL + 'products.csv', os.path.join(raw_dir, 'products.csv'))
    download_file(FILES_URL + 'matches.parquet', os.path.join(raw_dir, 'matches.parquet'))
    
    teams_df = pd.read_csv(raw_dir + '/' + 'teams.csv')
    products_df = pd.read_csv(raw_dir + '/' + 'products.csv')
    matches_df = pd.read_parquet(raw_dir + '/' +  'matches.parquet')

    orders_df = pd.json_normalize(orders, sep='_').rename(columns = {'customer_name': 'customer', 'customer_city': 'city'})

    return {'orders': orders_df, 'products': products_df, 'teams': teams_df, 'matches': matches_df}

In [0]:
# 6/18

def clean_products(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    is_dup = df.duplicated('product_id', keep='first')
    df = df[~is_dup]

    df['team_code'] = df['team_code'].str.strip().str.upper()
    df['category'] = df['category'].str.strip().str.lower()

    df['price'] = df['price'].str.replace('RSD', '', regex = False).str.replace(',', '.').str.strip()
    df['price'] = pd.to_numeric(df['price'])

    d = {'yes': True, 'Yes': True, 'YES': True}
    df['active'] = df['active'].map(d).fillna(False)

    return df

In [0]:
# 7/18

def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    for column in ['order_id', 'customer', 'city', 'payment']:
        df[column] = df[column].str.strip()

    df['order_id'] = df['order_id'].str.upper()
    df['customer'] = df['customer'].str.title()
    df['city'] = df['city'].str.title().fillna('Unknown')

    return df

In [0]:
# 8/18

def remove_duplicate_orders(df: pd.DataFrame) -> tuple:
    is_dup = df.duplicated('order_id', keep='first')
    return (df[~is_dup], df[is_dup])

In [0]:
# 9/18

def validate_order(row: dict, product_ids: set) -> str:
    quantity = pd.to_numeric(row['quantity'], errors='coerce')
    unit_price = pd.to_numeric(row['unit_price'], errors='coerce')
    
    if not row['order_id'].startswith('NSS-'):
        return 'invalid_order'
    if pd.isna(row['quantity']):
        return 'missing_quantity'
    if pd.isna(quantity) or quantity <= 0 or quantity != int(quantity):
        return 'invalid_quantity'
    if pd.isna(unit_price) or unit_price <= 0:
        return 'invalid_price'
    if row['product_id'] not in product_ids:
        return 'unknown_product'
     
    return 'ok'

In [0]:
# 10/18

def transform_orders(df: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    products = products.copy()
    # clean_products_df = clean_products(products)    #izbaciti?

    df['quantity'] = pd.to_numeric(df['quantity'], errors = 'coerce', downcast='integer')
    df['unit_price'] = pd.to_numeric(df['unit_price'], errors = 'coerce', downcast='float')
    df['order_date'] = pd.to_datetime(df['order_date'])
    df['total'] = df['quantity'] * df['unit_price']

    df.merge(products[['product_id', 'product_name', 'team_code', 'category']], how = 'left', on = 'product_id')

    return df

In [0]:
# 11/18

def split_orders(df: pd.DataFrame, products: pd.DataFrame) -> tuple:
    df = df.copy()
    products = products.copy()

    # clean_df = clean_orders(df) #izbaciti?
    unique_df, duplicate_df = remove_duplicate_orders(df)

    product_ids = set(products['product_id'])
    duplicate_df = duplicate_df.assign(reason='duplicate')

    reason = unique_df.apply(lambda order: validate_order(order, product_ids), axis=1)
    valid = transform_orders(unique_df[reason == 'ok'], products)
    rejected = pd.concat([unique_df[reason != 'ok'].assign(reason=reason[reason != 'ok']), duplicate_df]) 
    
    return valid, rejected

In [0]:
# 12/18

def team_results(matches: pd.DataFrame, teams: pd.DataFrame) -> pd.DataFrame:
    matches = matches[matches['status'] == 'finished']
    codes = dict(zip(teams['team_id'], teams['code']))

    home = pd.DataFrame({
        'match_id': matches['match_id'],
        'match_date': matches['kickoff_ts'].dt.normalize(),
        'team_code': matches['home_team_id'].map(codes),
        'opponent_code': matches['away_team_id'].map(codes),
        'goals_for': matches['home_goals'],
        'goals_against': matches['away_goals'],
        'results': matches.apply(lambda match: 'W' if match['home_goals'] > match['away_goals'] else 'D' if match['home_goals'] == match['away_goals'] else 'L', axis = 1),
        'points': matches.apply(lambda match: 3 if match['home_goals'] > match['away_goals'] else 1 if match['home_goals'] == match['away_goals'] else 0, axis = 1)
    })

    away = pd.DataFrame({
        'match_id': matches['match_id'],
        'match_date': matches['kickoff_ts'].dt.normalize(),
        'team_code': matches['away_team_id'].map(codes),
        'opponent_code': matches['home_team_id'].map(codes),
        'goals_for': matches['away_goals'],
        'goals_against': matches['home_goals'],
        'results': matches.apply(lambda match: 'W' if match['away_goals'] > match['home_goals'] else 'D' if match['home_goals'] == match['away_goals'] else 'L', axis = 1),
        'points': matches.apply(lambda match: 3 if match['away_goals'] > match['home_goals'] else 1 if match['away_goals'] == match['home_goals'] else 0, axis = 1)
    })

    return pd.concat([home, away], ignore_index= True)

In [0]:
# 13/18

# def league_table(results: pd.DataFrame) -> pd.DataFrame:




In [0]:
# 14/18

# def match_windows(results: pd.DataFrame) -> pd.DataFrame:



In [0]:
# 15/18

# def revenue_by(df: pd.DataFrame, column: str) -> pd.DataFrame:



In [0]:
# 16/18

# def sales_after_results(valid: pd.DataFrame, windows: pd.DataFrame) -> pd.DataFrame:
#     teams = pd.DataFrame({'team_code': windows['team_code'].unique()})
#     days = pd.DataFrame({'order_date': valid['order_date'].unique()})
#     calendar = teams.merge(days, how='cross')

#     valid.groupby(['team_code', 'order_date'])
#     valid['total'].sum().reset_index(name='revenue')

#     hmm = calendar.merge(windows, left_on=['team_code', 'order_date'], right_on=['team_code', 'date'], how='left')

#     return hmm


In [0]:
# 17/18

# def load(outputs: dict, processed_dir: str) -> dict:
#     written = {}
#     for name, df in outputs.items():
#         path = os.path.join(processed_dir, name)
#         if name.endswith('.parquet'):
#             df.to_parquet(path, index=False)
#         else:
#             df.to_csv(path, index=False)
#         written[name] = len(df)
#     return written

In [0]:
# 18/18

# def run_pipeline(raw_dir: str, processed_dir: str) -> dict:
#     data = extract(raw_dir)
#     products = clean_products(data['products'])
#     valid, rejected = split_orders(data['orders'], products)
#     results = team_results(data['matches'], data['teams'])
#     windows = match_windows(results)
#     outputs = {
#         'clean_orders.parquet': valid,
#         'rejected_orders.csv': rejected,
#         'products.csv': products,
#         'league_table.csv': league_table(results),
#         'revenue_by_team.csv': revenue_by(valid.dropna(subset=['team_code']), 'team_code'),
#         'revenue_by_category.csv': revenue_by(valid, 'category'),
#         'sales_after_results.csv': sales_after_results(valid, windows),
#         }
#     written = load(outputs, processed_dir)
#     summary = {
#         'total_rows': len(data['orders']),
#         'duplicates': int((rejected['reason'] == 'duplicate').sum()),
#         'valid_rows': len(valid),
#         'rejected_rows': len(rejected),
#         'total_revenue': round(float(valid['total'].sum()), 2),
#         'files_written': len(written) + 1,
#         }
#     pd.DataFrame(list(summary.items()), columns=['metric',
#     'value']).to_csv(
#         os.path.join(processed_dir, 'summary.csv'), index=False)
#     return summary

# display(run_pipeline(RAW_DIR, PROCESSED_DIR))